In [ ]:
# =============================================================================
# Synthetic curve dataset utility functions
# Includes: 3D visualization, camera projection, pseudo-image generation, rigid curve transforms, etc.
# =============================================================================

import numpy as np
import cv2
import plotly.graph_objs as go
import matplotlib.pyplot as plt
import os
import random  # random trace colors in show_interactive_3d
import open3d as o3d
from scipy.interpolate import splprep, splev

#-----------------------------------------------------------------------------------------------------------------
def show_interactive_3d(*args):
    """
    Interactive Plotly visualization of multiple 3D objects (point clouds, curves, camera poses).
    Args:
        *args: tuples of (data, mode, name[, color])
            - data: N×3 point set, or camera position/pose when mode='c'
            - mode: 'p' point cloud, 'l' polyline, 'c' camera (with XYZ axes)
            - name: legend label
            - color: optional color string
    """
    fig = go.Figure()
    for i, arg in enumerate(args):
        if len(arg) not in [3,4]:
            raise ValueError(f"Argument {i} should be tuple of (data, mode, name[, color])")

        data, mode, name = arg[:3]
        color = arg[3] if len(arg) == 4 else None
            
        if mode in ['p', 'l']:
            data = np.array(data)
            if data.ndim != 2 or data.shape[1] != 3:
                raise ValueError(f"Argument {i} ({name}): data should be (N,3) array for mode '{mode}'")
        elif mode == 'c':
            pass
        else:
            raise ValueError(f"Argument {i} ({name}): unknown mode '{mode}'")

        if color is None:
            color = "rgb({}, {}, {})".format(
                random.randint(50, 255),
                random.randint(50, 255),
                random.randint(50, 255)
            )
        
        if mode == 'p':
            fig.add_trace(go.Scatter3d(
                x=data[:, 0], y=data[:, 1], z=data[:, 2],
                mode='markers',
                marker=dict(size=2, color=color),
                name=name
            ))
        elif mode == 'l':
            fig.add_trace(go.Scatter3d(
                x=data[:, 0], y=data[:, 1], z=data[:, 2],
                mode='lines',
                line=dict(color=color, width=2),
                name=name
            ))
        else:  # mode == 'c': draw camera pose and local frame (red/green/blue = X/Y/Z)
            if isinstance(data, tuple) and len(data) == 2:
                pos, R = data
                pos = np.array(pos).reshape(3)
                R = np.array(R).reshape(3,3)
            else:
                pos = np.array(data).reshape(3)
                R = np.eye(3)

            if color is None:
                color = "rgb({}, {}, {})".format(
                    random.randint(50, 255),
                    random.randint(50, 255),
                    random.randint(50, 255)
                )
            
            fig.add_trace(go.Scatter3d(
                x=[pos[0]], y=[pos[1]], z=[pos[2]],
                mode='markers+text',
                marker=dict(size=2, color=color, symbol='circle'),
                text=[name],
                textposition='top center',
                name=name
            ))

            axis_length = 4.0
            axes_colors = ['red', 'green', 'blue']  # X, Y, Z
            for j in range(3):
                start = pos
                end = pos + R[:, j] * axis_length
                fig.add_trace(go.Scatter3d(
                    x=[start[0], end[0]],
                    y=[start[1], end[1]],
                    z=[start[2], end[2]],
                    mode='lines',
                    line=dict(color=axes_colors[j], width=4),
                    name=f'{name} axis {["X","Y","Z"][j]}'
                ))
    fig.update_layout(
        scene=dict(
        xaxis=dict(title='X', range=[-50, 50]),
        yaxis=dict(title='Y', range=[-50, 50]),
        zaxis=dict(title='Z', range=[-10, 90]),
        aspectmode='manual',
        aspectratio=dict(x=1, y=1, z=1)
        ),
        showlegend=True
    )
    fig.show()

#-----------------------------------------------------------------------------------------------------------------
def project_curve_to_image(points_3d, K, R, t):
    """
    Project 3D points in world coordinates onto the camera image plane.

    Args:
        points_3d: N×3 world coordinates
        K: 3×3 intrinsic matrix
        R, t: camera rotation and translation in world frame (R is camera→world, t is camera center)

    Returns:
        points_2d: N×2 pixel coordinates
        depths: N depths (camera-frame Z)
    """
    if t.ndim == 1:
        t = t.reshape(3, 1)

    # Convert camera→world extrinsics to world→camera
    R_wc = R.T
    t_wc = -R.T @ t

    # Projection matrix P = K [R_wc | t_wc]
    Rt = np.hstack((R_wc, t_wc))  # 3×4
    P = K @ Rt

    # Homogeneous projection
    N = points_3d.shape[0]
    points_3d_hom = np.hstack((points_3d, np.ones((N, 1))))  # N×4
    points_2d_hom = (P @ points_3d_hom.T).T  # N×3

    # Perspective division to pixel coordinates
    points_2d = points_2d_hom[:, :2] / points_2d_hom[:, 2:3]
    depths = points_2d_hom[:, 2]

    return points_2d, depths

#-----------------------------------------------------------------------------------------------------------------



def generate_pseudo_image_for_CPlusPlus(points, img_size=(640,480), noise_std=10, ksize=(3,3)):
    """
    Generate a pseudo camera image: draw white points at pixel locations, optionally add Gaussian noise.

    Args:
        points: N×2 pixel coordinates (x, y)
        img_size: (width, height)
        noise_std: Gaussian noise std dev; 0 disables noise
        ksize: Gaussian kernel size (must be odd; blur not applied in current implementation)

    Returns:
        img_rgb: uint8 RGB image for C++ consumption
    """
    assert ksize[0] % 2 != 0 and ksize[1] % 2 != 0, "Gaussian kernel size must be odd."

    img_w, img_h = img_size
    img = np.zeros((img_h, img_w), dtype=np.float32)

    # Quantize projected points to pixel grid and filter out-of-bounds points
    pixels = np.round(points).astype(int)
    mask = (pixels[:, 0] >= 0) & (pixels[:, 0] < img_w) & \
           (pixels[:, 1] >= 0) & (pixels[:, 1] < img_h)
    pixels = pixels[mask]

    if pixels.size > 0:
        img[pixels[:, 1], pixels[:, 0]] = 255  # OpenCV indexing is (row, col) = (y, x)

    if noise_std > 0:
        noise = np.random.normal(0, noise_std, img.shape).astype(np.float32)
        img += noise

    img_uint8 = np.clip(img, 0, 255).astype(np.uint8)
    img_rgb = cv2.merge([img_uint8]*3)  # Replicate grayscale to three channels

    return img_rgb


def get_rotation_matrix(x_rot, y_rot, z_rot):
    """Rotate about X/Y/Z axes (degrees); return combined rotation R = Rz @ Ry @ Rx."""
    theta_x = np.deg2rad(x_rot)
    theta_y = np.deg2rad(y_rot)
    theta_z = np.deg2rad(z_rot)

    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(theta_x), -np.sin(theta_x)],
        [0, np.sin(theta_x),  np.cos(theta_x)]
    ])
    Ry = np.array([
        [np.cos(theta_y), 0, np.sin(theta_y)],
        [0, 1, 0],
        [-np.sin(theta_y), 0, np.cos(theta_y)]
    ])
    Rz = np.array([
        [np.cos(theta_z), -np.sin(theta_z), 0],
        [np.sin(theta_z),  np.cos(theta_z), 0],
        [0,                0,               1]
    ])

    return Rz @ Ry @ Rx

#-----------------------------------------------------------------------------------------------------------------
def apply_rotation_translation(curve, target_point, translation, rotation):
    """Apply rigid transform to curve about target_point as pivot: rotate then translate."""
    curve_shifted = curve - target_point
    R = get_rotation_matrix(*rotation)
    return (R @ curve_shifted.T).T + target_point + translation

In [ ]:
# =============================================================================
# Stereo camera setup (orbital mode from Triangulation Made Easy)
# =============================================================================

# Target distance from baseline center = 2^n_d * baseline
# n_d = -1, 0, ..., +8 controls viewing distance; n_d=2 → distance = 4 * baseline
n_d = 2

# ---------- Camera intrinsics ----------
fx, fy = 512, 512
cx, cy = 512, 512
K = np.array([
    [fx, 0, cx],
    [0, fy, cy],
    [0,  0,  1]
])

# ---------- Baseline and target point ----------
baseline = 10
dis_from_baseline = 2 ** n_d * baseline
target_point = np.array([0, 0, dis_from_baseline])  # target point viewed by both cameras

# Left/right camera centers (symmetric about X axis)
cam_position1 = np.array([-0.5 * baseline, 0, 0])
cam_position2 = np.array([0.5 * baseline, 0, 0])

# ---------- Camera extrinsics: rotate about Y so optical axis points at target_point ----------
direction = target_point - cam_position1
theta_y = np.arctan2(direction[0], direction[2])
c, s = np.cos(theta_y), np.sin(theta_y)
R1 = np.array([
    [ c, 0, s],
    [ 0, 1, 0],
    [-s, 0, c]
])
direction = target_point - cam_position2
theta_y = np.arctan2(direction[0], direction[2])
c, s = np.cos(theta_y), np.sin(theta_y)
R2 = np.array([
    [ c, 0, s],
    [ 0, 1, 0],
    [-s, 0, c]
])

t1 = cam_position1.reshape(3,1)
t2 = cam_position2.reshape(3,1)

# ---------- Export quaternion + translation (for SLAM/VINS C++ pipelines) ----------
# Format: QW, QX, QY, QZ, TX, TY, TZ
from scipy.spatial.transform import Rotation as R

def rotation_to_quaternion(R_mat):
    """Rotation matrix → quaternion (QW, QX, QY, QZ); scipy returns (x,y,z,w)."""
    r = R.from_matrix(R_mat)
    q = r.as_quat()
    QW = q[3]
    QX = q[0]
    QY = q[1]
    QZ = q[2]
    return QW, QX, QY, QZ

# Left camera (cam1)
QW1, QX1, QY1, QZ1 = rotation_to_quaternion(R1)
TX1, TY1, TZ1 = t1.flatten()

# Right camera (cam2)
QW2, QX2, QY2, QZ2 = rotation_to_quaternion(R2)
TX2, TY2, TZ2 = t2.flatten()

In [ ]:
# =============================================================================
# Eight 3D curve generators
# Naming: curve_id(01-08) + param(0-2) + translation(0-3) + rotation(0-3)
# All curves are generated in the XY plane, shifted to target_point, then optionally rigidly transformed
# =============================================================================

# 01 - Circle (half-circle sampling, theta ∈ [-π, 0.95π])
def circle(target_point, N, param, translation=np.zeros(3), rotation=(0,0,0)):
    param_map = {
    "0": 10,   # radius
    "1": 5,
    "2": 15}
    radius = param_map[param]
    
    n = np.linspace(-1, 0.95, N)
    theta = np.pi * n
    x = radius * np.cos(theta) + target_point[0]
    y = radius * np.sin(theta) + target_point[1]
    z = np.full_like(n, target_point[2])
    curve = np.vstack((x, y, z)).T
    return apply_rotation_translation(curve, target_point, translation, rotation)


# 02 - Ellipse (a, b = semi-major / semi-minor axes)
def ellipse(target_point, N, param, translation=np.zeros(3), rotation=(0,0,0)):
    param_map = {
    "0": (15, 10),
    "1": (15, 7),
    "2": (15, 13)}
    a, b = param_map[param]
    n = np.linspace(-1, 0.95, N)
    theta = np.pi * n
    x = a * np.cos(theta) + target_point[0]
    y = b * np.sin(theta) + target_point[1]
    z = np.full_like(n, target_point[2])
    curve = np.vstack((x, y, z)).T
    return apply_rotation_translation(curve, target_point, translation, rotation)

# 03 - Parabola y = a*n^2 + c
def parabola(target_point, N, param, translation=np.zeros(3), rotation=(0,0,0)):
    param_map = {
    "0": (-30, 15),
    "1": (-40, 15),
    "2": (-20, 15)}
    a, c = param_map[param]
    n = np.linspace(-0.85, 0.85, N)
    x = 15 * n + target_point[0]
    y = a * n**2 + c + target_point[1]
    z = np.full_like(n, target_point[2])
    curve = np.vstack((x, y, z)).T
    return apply_rotation_translation(curve, target_point, translation, rotation)

# 04 - Hyperbola y = a/n + b (n must not be 0)
def hyperbola(target_point, N, param, translation=np.zeros(3), rotation=(0,0,0)):
    param_map = {
    "0": (0.3, 8, -8, 10),
    "1": (0.17, 8, -5, 10),
    "2": (0.92, 8, -30, 10)}
    n_min, n_max, a, b = param_map[param]
    n = np.linspace(n_min, n_max, N)
    x = 3 * n - 10 + target_point[0]
    y = a / n + b + target_point[1]
    z = np.full_like(n, target_point[2])
    curve = np.vstack((x, y, z)).T
    return apply_rotation_translation(curve, target_point, translation, rotation)

# 05 - Planar Archimedean spiral r = a + b*θ
def spiral(target_point, N, param, translation=np.zeros(3), rotation=(0,0,0)):
    param_map = {
    "0": (2.0, -2.0, -3.0),
    "1": (4, -1.5, -1.5),
    "2": (1.5, 3.0, 3.6)}
    t, a, b = param_map[param]
    theta_max = t * np.pi
    theta = np.linspace(0, theta_max, N)
    r = a + b * theta
    x = r * np.cos(theta) + target_point[0]
    y = r * np.sin(theta) + target_point[1]
    z = np.full_like(theta, target_point[2])
    curve = np.vstack((x, y, z)).T
    return apply_rotation_translation(curve, target_point, translation, rotation)

# 06 - B-spline curve (splprep/splev interpolation through control points)
def spline_curve(target_point, N, param, translation=np.zeros(3), rotation=(0,0,0)):
    param_map = {
    "0": np.array([
        [-20,  0],
        [-5,   3],
        [ 0,  -1],
        [ 6,   4],
        [11,   2],
        [15,   0]]),
    "1": np.array([
        [-20,  0],
        [-5,   -2],
        [ 0,  3],
        [ 8,   0],
        [15,   2]]),
    "2": np.array([
        [-21,  0],
        [-16,  5],
        [-11,   2],
        [ -6,  5],
        [ -1,   2],
        [4,   5],
        [9,   0]])}
    control_points = param_map[param]
    ctrl_x = control_points[:,0]
    ctrl_y = control_points[:,1]
    tck, u = splprep([ctrl_x, ctrl_y], s=0)
    u_fine = np.linspace(0, 1, N)
    x, y = splev(u_fine, tck)
    z = np.full_like(x, target_point[2])
    curve = np.vstack((x + target_point[0], y + target_point[1], z)).T
    return apply_rotation_translation(curve, target_point, translation, rotation)

# 07 - Composite curves (semicircle+line / line+arc+line / two arcs+line; param selects layout)
def composite_curve(target_point, N, param, translation=np.zeros(3), rotation=(0,0,0)):
    n = np.linspace(-1,1,N)
    
    if param == "0":
        # ---------------- 1. Semicircle ----------------
        theta = np.linspace(-0.5 * np.pi, 0.5 * np.pi, N//2)  # semicircle angles from -π/2 to π/2
        radius = 10
        x_circle = 5 + radius * np.cos(theta) + target_point[0]
        y_circle = radius * np.sin(theta) + target_point[1]

        # ---------------- 2. Line segment (tangent to semicircle end) ----------------
        x_start = x_circle[-1]  # semicircle end X
        y_start = y_circle[-1]  # semicircle end Y

        line_length = 20
        x_line = np.linspace(x_start, x_start - line_length, N//2)
        y_line = np.full_like(x_line, y_start)  # horizontal line, tangent to semicircle end

        # ---------------- Merge ----------------
        y = np.concatenate([x_circle, x_line])
        x = np.concatenate([y_circle, y_line])
        z = np.full_like(x, target_point[2])  # Z unchanged
    elif param == "1":
        N_line1 = int(N * 80 / 300)   # ≈ 13333
        N_arc   = int(N * 140 / 300)  # ≈ 23333
        N_line2 = N - N_line1 - N_arc # 50000 - 13333 - 23333 = 13334
        radius = 10  # arc radius
        # ---------------- 1. First line segment (along X axis) ----------------
        x_line1 = np.linspace(0, 20, N_line1)
        y_line1 = np.zeros_like(x_line1)
        # ---------------- 2. Circular arc (120 deg = 2pi/3 rad) ----------------
        theta_arc = np.linspace(0, 2*np.pi/3, N_arc)
        # Arc center above/right of first segment end for tangency
        center_x = x_line1[-1]
        center_y = radius  # enforce horizontal tangent
        x_arc = center_x + radius * np.sin(theta_arc)  # use sin for horizontal tangent at start
        y_arc = center_y - radius * np.cos(theta_arc)  # tangent to horizontal first segment
        # ---------------- 3. Second line segment (tangent to arc end) ----------------
        # compute tangent direction at arc end
        dx = x_arc[-1] - x_arc[-2]
        dy = y_arc[-1] - y_arc[-2]
        length_line2 = 20
        x_line2 = np.linspace(x_arc[-1], x_arc[-1] + length_line2 * (dx/np.hypot(dx,dy)), N_line2)
        y_line2 = np.linspace(y_arc[-1], y_arc[-1] + length_line2 * (dy/np.hypot(dx,dy)), N_line2)
        # ---------------- Merge ----------------
        x = np.concatenate([x_line1, x_arc, x_line2]) - 15
        y = np.concatenate([y_line1, y_arc, y_line2]) - 10
        z = np.full_like(x, target_point[2])
    elif param == "2":
        N_arc1 = int(N * 140 / 300)   # first arc segment
        N_line = int(N * 80 / 300)    # middle line segment
        N_arc2 = N - N_arc1 - N_line  # second arc segment
        radius = 10  # arc radius
        # ---------------- First arc segment ----------------
        theta1 = np.linspace(0, 2*np.pi/3, N_arc1)  # 120deg
        cx1, cy1 = 0, radius
        x_arc1 = cx1 + radius * np.sin(theta1)
        y_arc1 = cy1 - radius * np.cos(theta1)
        # tangent direction at end
        dx1 = x_arc1[-1] - x_arc1[-2]
        dy1 = y_arc1[-1] - y_arc1[-2]
        tangent1 = np.array([dx1, dy1])
        tangent1 /= np.linalg.norm(tangent1)
        # ---------------- Middle line segment ----------------
        line_length = 20
        x_line = np.linspace(x_arc1[-1], x_arc1[-1] + line_length * tangent1[0], N_line)
        y_line = np.linspace(y_arc1[-1], y_arc1[-1] + line_length * tangent1[1], N_line)
        # ---------------- Second arc segment ----------------
        # arc center: radius away, perpendicular to line direction
        perp = np.array([-tangent1[1], tangent1[0]])
        cx2, cy2 = x_line[-1] + perp[0]*radius, y_line[-1] + perp[1]*radius
        # arc angles so start tangent matches line direction
        theta2_start = np.arctan2(y_line[-1]-cy2, x_line[-1]-cx2)
        theta2 = np.linspace(theta2_start, theta2_start + 2*np.pi/3, N_arc2)
        x_arc2 = cx2 + radius * np.cos(theta2)
        y_arc2 = cy2 + radius * np.sin(theta2)
        # ---------------- Merge ----------------
        x = np.concatenate([x_arc1, x_line, x_arc2]) + 3
        y = np.concatenate([y_arc1, y_line, y_arc2]) - 18  
        z = np.full_like(x, target_point[2])   
    else:
        return

    curve = np.vstack((x, y, z)).T
    return apply_rotation_translation(curve, target_point, translation, rotation)

# 08 - Spatial spiral: planar spiral + linear Z variation
def spatial_spiral(target_point, N, param, translation=np.zeros(3), rotation=(0,0,0)):
    param_map = {
    "0": 2,
    "1": -2,
    "2": -4}
    k = param_map[param] # Z slope coefficient (planar spiral uses fixed a, b below)
    t = 3.0
    a = 2.0
    b = 1.5
    n = np.linspace(-1.0, 1.0, N)
    theta_max = t * np.pi  # total rotation angle
    theta = np.linspace(0, theta_max, N)
    r = a + b * theta
    x = r * np.cos(theta) + target_point[0]
    y = r * np.sin(theta) + target_point[1]
    z = k * n + target_point[2]
    curve = np.vstack((x, y, z)).T
    return apply_rotation_translation(curve, target_point, translation, rotation)

# ---------- Curve types and variation dictionaries ----------
curve_types = {
    "01": circle,
    "02": ellipse,
    "03": parabola,
    "04": hyperbola,
    "05": spiral,
    "06": spline_curve,
    "07": composite_curve,
    "08": spatial_spiral
}

param_types = {
    "0": "0",  # first geometry parameter set for each curve type
    "1": "1",
    "2": "2"
}

translation_types = {
    "0": np.array([0.0, 0.0, 0.0]),
    "1": np.array([15.0, 0.0, 0.0]),   # translate along X
    "2": np.array([0.0, 15.0, 0.0]),   # translate along Y
    "3": np.array([0.0, 0.0, 15.0])    # translate along Z
}

rotation_types = {
    "0": (0.0, 0.0, 0.0),
    "1": (45.0, 0.0, 0.0),   # rotate 45° about X
    "2": (0.0, 45.0, 0.0),   # rotate 45° about Y
    "3": (0.0, 0.0, 45.0)    # rotate 45° about Z
}


# ---------- Global dataset parameters ----------
N = 50000                # number of sample points per curve
img_width = 1024
img_height = 1024

root_path = "./"
if not os.path.exists(root_path):
    os.makedirs(root_path)

# Single-curve debug example (uncomment to run):
# curve_3d = curve_types["01"](target_point, N, param_types["0"], translation_types["0"], rotation_types["0"])
# point_cloud_gt = curve_3d.astype(np.float32)
# pcd = o3d.geometry.PointCloud()
# pcd.points = o3d.utility.Vector3dVector(point_cloud_gt)
# o3d.io.write_point_cloud(root_path + "point_cloud.ply", pcd)

In [ ]:
# =============================================================================
# Batch-generate synthetic dataset
# 8×3×4×4 = 384 cases; each case contains:
#   - pointclouds/{case_name}.ply  3D curve ground-truth point cloud
#   - images/{case_name}/1.png     left camera pseudo image
#   - images/{case_name}/2.png     right camera pseudo image
# case_name format: {curve_id}_{param}_{trans}_{rot}, e.g. "01_0_0_0"
# =============================================================================

for curve_id, curve_func in curve_types.items():
    for param_id, param_value in param_types.items():
        for trans_id, trans_value in translation_types.items():
            for rot_id, rot_value in rotation_types.items():

                # 1. Generate 3D curve
                curve_3d = curve_func(
                    target_point, N,
                    param = param_value,
                    translation = trans_value,
                    rotation = rot_value)

                print(f"Case[{curve_id}_{param_id}_{trans_id}_{rot_id}]")
                case_name = curve_id + "_" + param_id + "_" + trans_id + "_" + rot_id

                # 2. Save ground-truth point cloud (.ply)
                point_cloud = curve_3d.astype(np.float32)
                pcd = o3d.geometry.PointCloud()
                pcd.points = o3d.utility.Vector3dVector(point_cloud)
                pointcloud_name = case_name + ".ply"

                pointcloud_path = root_path + "pointclouds/"
                if not os.path.exists(pointcloud_path):
                    os.makedirs(pointcloud_path)
                o3d.io.write_point_cloud(pointcloud_path + pointcloud_name, pcd)
                print(f"Saved {pointcloud_name}")

                # 3. Project to left/right cameras and generate pseudo images
                points_2D1_ori, depths_2D1_ori = project_curve_to_image(curve_3d, K, R1, t1)
                points_2D2_ori, depths_2D2_ori = project_curve_to_image(curve_3d, K, R2, t2)

                img1 = generate_pseudo_image_for_CPlusPlus(points_2D1_ori, img_size=(img_width, img_height), noise_std=0, ksize=(1,1))
                img2 = generate_pseudo_image_for_CPlusPlus(points_2D2_ori, img_size=(img_width, img_height), noise_std=0, ksize=(1,1))

                img_path = root_path + "images/" + case_name + "/"
                if not os.path.exists(img_path):
                    os.makedirs(img_path)
                cv2.imwrite(img_path + "1.png", img1)
                cv2.imwrite(img_path + "2.png", img2)
                print(f"Saved 1.png, 2.png")

                # 4. Visualize stereo pseudo images for current case
                plt.figure(figsize=(6,3))
                plt.subplot(1,2,1)
                plt.imshow(img1)
                plt.title("Pseudo Image 1")
                plt.axis("off")

                plt.subplot(1,2,2)
                plt.imshow(img2)
                plt.title("Pseudo Image 2")
                plt.axis("off")
                plt.show()